# 🎮 Proyecto Street Fighter con MediaPipe Hands  

Nuestro proyecto consiste en una **versión interactiva de Street Fighter** que utiliza la tecnología de **MediaPipe Hands** para controlar los movimientos de los personajes mediante gestos con las manos.  

En lugar de utilizar un control tradicional, los jugadores podrán **pelear con movimientos físicos**, haciendo la experiencia más inmersiva y dinámica.  

## ⚙️ ¿Cómo funciona?  
- A partir de estos puntos, el sistema interpreta gestos específicos para transformarlos en acciones dentro del juego.  

## 🕹️ Movimientos implementados  
- ✊ **Golpe** → Movimiento de puño cerrado.  
- ✋ **Avanzar** → Gesto de mano hacia adelante.  
- 🖐️ **Saltar** → Gesto de mano levantada.  
- 🦵 **Patada** → Combinación de gestos que activan la acción de patear.  


In [37]:
import mediapipe as mp
print(mp.__version__)


0.10.21


## ⭕ MediaPipe Pose - Skeleton KeyPoint 

<img style="padding-left:50px;" src="figuras/MediaPipePose.jpg">




In [38]:
def limites_saltar_agachar(y_nariz, y_hombro_izq, y_hombro_der):
    y_delta_hombro_nariz = abs(y_nariz - ((y_hombro_izq + y_hombro_der) / 2))
    
    limite_saltar = y_nariz - y_delta_hombro_nariz
    limite_agachar = 1.33 * (y_nariz + y_delta_hombro_nariz)
    
    return limite_saltar, limite_agachar

In [39]:
def limites_acerca_alejar(x_nariz, x_hombro_izq, x_hombro_der):
    x_hombro = (x_hombro_izq + x_hombro_der) / 2
    x_delta_hombro_nariz = abs(x_nariz - (x_hombro))
    
    limite_acercar = 0.8 * (x_hombro + x_delta_hombro_nariz)
    limite_alejar = 1.4 * (x_hombro - x_delta_hombro_nariz)
    
    return limite_acercar, limite_alejar

In [40]:
def limite_golpe_der(y_hombro_der, y_mano_der):
    longitud_brazo = abs(y_hombro_der - y_mano_der)
    x_limite_golpear = 0.55 * longitud_brazo
    return x_limite_golpear

In [41]:
def limite_golpe_izq(y_hombro_izq, y_mano_izq):
    longitud_brazo = abs(y_hombro_izq - y_mano_izq)
    x_limite_golpear = 0.55 * longitud_brazo
    return x_limite_golpear


In [ ]:
import cv2
import mediapipe as mp
import pydirectinput

# --- Inicializar MediaPipe Pose (para cuerpo) ---
mp_pose = mp.solutions.pose
pose = mp_pose.Pose()
mp_drawing_pose = mp.solutions.drawing_utils

# --- Helpers de cámara ---
def open_cam(idx):
    cap = cv2.VideoCapture(idx, cv2.CAP_DSHOW)
    if not cap.isOpened():
        return None
    ok, _ = cap.read()
    if not ok:
        cap.release()
        return None
    return cap

# --- Abre cámara inicial (0; si no, 1) ---
current_idx = 0
cap = open_cam(current_idx)
if cap is None:
    current_idx = 0
    cap = open_cam(current_idx)
if cap is None:
    raise RuntimeError("No se pudo abrir cámara 0 ni 1.")

# --- Variable para acción corporal ---
accion = "Neutra"
proporcion_inicial_flag = False

limite_saltar = 0
limite_agachar = 0

limite_acercar = 0
limite_alejar = 0
x_limite_golpear = 0

estado_teclas = {'up': False, 'down': False, 'left': False, 'right': False, 'alt': False, 'ctrlleft': False, 'z': False, 'x': False, 'hadouken': False}

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Copia de frame original
    img = frame.copy()
    fraccion = 1
    image = cv2.resize(img, (0, 0), fx=fraccion, fy=fraccion, interpolation=cv2.INTER_NEAREST)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    
    image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)

    # Procesar pose y manos
    results_pose = pose.process(image_rgb)

    if results_pose.pose_landmarks:

        y_nariz = results_pose.pose_landmarks.landmark[0].y
        y_hombro_izq = results_pose.pose_landmarks.landmark[11].y
        y_hombro_der = results_pose.pose_landmarks.landmark[12].y

        x_nariz = results_pose.pose_landmarks.landmark[0].x
        x_hombro_izq = results_pose.pose_landmarks.landmark[11].x
        x_hombro_der = results_pose.pose_landmarks.landmark[12].x
        
        y_mano_der = results_pose.pose_landmarks.landmark[20].y
        x_mano_der = results_pose.pose_landmarks.landmark[20].x
        y_mano_izq = results_pose.pose_landmarks.landmark[19].y
        x_mano_izq = results_pose.pose_landmarks.landmark[19].x
        
        x_pie_der = results_pose.pose_landmarks.landmark[28].x
        x_pie_izq = results_pose.pose_landmarks.landmark[27].x
        
    if proporcion_inicial_flag == False:

        limite_saltar, limite_agachar = limites_saltar_agachar(y_nariz, y_hombro_izq, y_hombro_der)
        limite_acercar, limite_alejar = limites_acerca_alejar(x_nariz, x_hombro_izq, x_hombro_der)
        x_limite_golpear_der = limite_golpe_der(y_hombro_der, y_mano_der)
        x_limite_golpear_izq = limite_golpe_izq(y_hombro_izq, y_mano_izq)
        proporcion_inicial_flag = True
        
        h, w = image_bgr.shape[:2]

        # Jump limit line (green)
        y_js = limite_saltar
        if y_js < 0.0: y_js = 0.0
        if y_js > 1.0: y_js = 1.0
        y_js_px = int(y_js * h)
        cv2.line(image_bgr, (0, y_js_px), (w - 1, y_js_px), (0, 255, 0), 2)

        # Crouch limit line (red)
        y_ag = limite_agachar
        if y_ag < 0.0: y_ag = 0.0
        if y_ag > 1.0: y_ag = 1.0
        y_ag_px = int(y_ag * h)
        cv2.line(image_bgr, (0, y_ag_px), (w - 1, y_ag_px), (0, 0, 255), 2)


    # ==================== DETECCIÓN DE POSE ====================
    if results_pose.pose_landmarks:
        mp_drawing_pose.draw_landmarks(image_bgr, results_pose.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        
         # === Dibujar los números de los landmarks ===
        h, w, _ = image_bgr.shape
        for idx, landmark in enumerate(results_pose.pose_landmarks.landmark):
            x = int(landmark.x * w)
            y = int(landmark.y * h)
            cv2.putText(image_bgr, str(idx), (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 
                        0.4, (0, 255, 255), 1, cv2.LINE_AA)
                 
        # --- Calcular límite de golpe dinámico ---
        x_limite_para_golpe_der = x_hombro_der - x_limite_golpear_der
        x_limite_para_golpe_izq = x_hombro_izq - x_limite_golpear_izq
        h, w = image_bgr.shape[:2]
        x_golpe_px = int(x_limite_para_golpe_der * w)
        cv2.line(image_bgr, (x_golpe_px, 0), (x_golpe_px, h), (255, 0, 0), 2)  # azul
        x_golpe_px2 = int(x_limite_para_golpe_izq * w)
        cv2.line(image_bgr, (x_golpe_px2, 0), (x_golpe_px2, h), (0, 0, 255), 2)  # azul
        
        # --- Dibujar línea de golpe (azul) ---
        nariz_y = results_pose.pose_landmarks.landmark[0].y  # coordenada Y de la nariz
        x11 = results_pose.pose_landmarks.landmark[11].x
        x12 = results_pose.pose_landmarks.landmark[12].x
        x_hombro = (x11 + x12) / 2

        # --- Movimiento vertical (saltar / agacharse) ---
        if y_nariz < limite_saltar:
            if not estado_teclas['up']:
                pydirectinput.keyDown('up')
                estado_teclas['up'] = True
            if estado_teclas['down']:
                pydirectinput.keyUp('down')
                estado_teclas['down'] = False
            accion = "Saltar"

        elif y_nariz > limite_agachar:
            if not estado_teclas['down']:
                pydirectinput.keyDown('down')
                estado_teclas['down'] = True
            if estado_teclas['up']:
                pydirectinput.keyUp('up')
                estado_teclas['up'] = False
            accion = "Agachar"

        else:
            if estado_teclas['up']:
                pydirectinput.keyUp('up')
                estado_teclas['up'] = False
            if estado_teclas['down']:
                pydirectinput.keyUp('down')
                estado_teclas['down'] = False
            accion = "Neutra"

        # --- Movimiento horizontal (acercar / alejar) ---
        if x_hombro < limite_acercar:
            if not estado_teclas['right']:
                pydirectinput.keyDown('right')
                estado_teclas['right'] = True
            if estado_teclas['left']:
                pydirectinput.keyUp('left')
                estado_teclas['left'] = False
            accion = "Acercar"

        elif x_hombro > limite_alejar:
            if not estado_teclas['left']:
                pydirectinput.keyDown('left')
                estado_teclas['left'] = True
            if estado_teclas['right']:
                pydirectinput.keyUp('right')
                estado_teclas['right'] = False
            accion = "Alejar"

        else:
            if estado_teclas['left']:
                pydirectinput.keyUp('left')
                estado_teclas['left'] = False
            if estado_teclas['right']:
                pydirectinput.keyUp('right')
                estado_teclas['right'] = False
            accion = "Neutra"
            
        # --- Golpes Especiales ---
        
        if x_mano_der < x_limite_para_golpe_der and x_mano_izq < x_limite_para_golpe_izq:
        # Ejecutar Hadouken
            if not estado_teclas['hadouken']:
                
                # Secuencia ↓ →↘ + puño (ctrlleft)
                pydirectinput.keyDown('down')
                pydirectinput.keyDown('right')
                pydirectinput.keyUp('down')
                pydirectinput.keyDown('ctrlleft')
                pydirectinput.keyUp('right')
                pydirectinput.keyUp('ctrlleft')

                estado_teclas['hadouken'] = True
            accion = "Hadouken"
        else:
            # Reset Hadouken
            estado_teclas['hadouken'] = False

         # --- Golpes ---
        
        if x_mano_der < x_limite_para_golpe_der:
            # Si la mano pasa el límite, presiona ALT una vez
            if not estado_teclas['alt']:
                pydirectinput.keyDown('alt')
                estado_teclas['alt'] = True
            accion = "Golpear - Alt"
            
        else:
            # Si la mano vuelve atrás, suelta ALT
            if estado_teclas['alt']:
                pydirectinput.keyUp('alt')
                estado_teclas['alt'] = False
            
        if x_mano_izq < x_limite_para_golpe_izq:
            # Si la mano pasa el límite, presiona Ctrl-Left una vez
            if not estado_teclas['ctrlleft']:
                pydirectinput.keyDown('ctrlleft')
                estado_teclas['ctrlleft'] = True
            accion = "Golpear - Ctrl-left"
        else:
            # Si la mano vuelve atrás, suelta Ctrl-Left
            if estado_teclas['ctrlleft']:
                pydirectinput.keyUp('ctrlleft')
                estado_teclas['ctrlleft'] = False
        
         # --- Golpes Patadas ---
        
        if x_pie_izq < x_limite_para_golpe_izq:
            # Si la mano pasa el límite, presiona Z una vez
            if not estado_teclas['z']:
                pydirectinput.keyDown('z')
                estado_teclas['z'] = True
            accion = "Patear - Z"
        else:
            # Si la mano vuelve atrás, suelta Z
            if estado_teclas['z']:
                pydirectinput.keyUp('z')
                estado_teclas['z'] = False
                
        if x_pie_der < x_limite_para_golpe_der:
            # Si la mano pasa el límite, presiona Z una vez
            if not estado_teclas['x']:
                pydirectinput.keyDown('x')
                estado_teclas['x'] = True
            accion = "Patear - X"
        else:
            # Si la mano vuelve atrás, suelta Z
            if estado_teclas['x']:
                pydirectinput.keyUp('x')
                estado_teclas['x'] = False
                
    # ==================== MOSTRAR EN PANTALLA ====================
    texto = "..."
    image_flipped = cv2.flip(image_bgr, 1)
    cv2.putText(image_flipped, f"Jugador: {texto}", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    
    cv2.putText(image_flipped, f"Accion cuerpo: {accion}", (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.imshow("Pose + Hand Tracking", image_flipped)


    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

NameError: name 'h' is not defined